# AREx on Colab: Transformers/A100
Run cells in order. The model path and ID are explicit overrides below.
Results, logs, and checkpoints persist in Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess
from pathlib import Path
REPO = Path('/content/AREx-source')
BRANCH = os.environ.get(
    'AREX_GIT_REF', 'codex/centralized-runtime-config')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH,
        'https://github.com/SiddarthaKoppaka/AREx', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', BRANCH], check=True)
    subprocess.run(
        ['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
print(subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD']).decode())


In [ ]:
import shutil, subprocess, sys
from pathlib import Path
REPO = Path('/content/AREx-source')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV = shutil.which('uv')
assert UV, 'uv installation failed'
subprocess.run([UV, 'python', 'install', '3.12'], check=True)
subprocess.run(
    [UV, 'venv', '/content/arex-py312', '--python', '3.12'], check=True)
subprocess.run([UV, 'pip', 'install', '--python',
    '/content/arex-py312/bin/python',
    '-e', f'{REPO}[colab]'], check=True)


In [ ]:
import subprocess
DATASET_ROOT = '/content/drive/MyDrive/AREx/environment_files'
subprocess.run([
    '/content/arex-py312/bin/python', '-m',
    'arc_agi_3.research.arc_bootstrap',
    '--dataset-root', DATASET_ROOT, '--game-id', 'ls20', '--seed', '0',
], check=True)


In [ ]:
import json, subprocess
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'A CUDA GPU is required'
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name}; VRAM: {gpu.total_memory / 1024**3:.1f} GiB')
OVERRIDES = {
    'model_path': (
        '/content/drive/MyDrive/AREx/models/'
        'REPLACE_WITH_MODEL_DIRECTORY'),
    'model_name': 'REPLACE_WITH_MODEL_ID',
}
assert 'REPLACE_WITH' not in json.dumps(OVERRIDES)
subprocess.run([
    '/content/arex-py312/bin/python', '-m',
    'arc_agi_3.research.colab_entry',
    '--profile', 'colab_transformers',
    '--overrides-json', json.dumps(OVERRIDES),
    '--dataset-root', '/content/drive/MyDrive/AREx/environment_files',
    '--game-id', 'ls20', '--seed', '0',
], check=True)


Each run gets a unique ID under `/content/drive/MyDrive/AREx/runs`.
Inspect `events.jsonl`, `manifest.json`, `checkpoints/`, and `result.json`.
The console log is under `/content/drive/MyDrive/AREx/logs`.
